#### Load Dataset

In [1]:
import pandas as pd
df = pd.read_csv('sample_kinder_data.csv')

#### Clean Dataset

In [2]:
def clean_data(df):
    # Drop rows with missing target values
    df = df.dropna(subset=['score_read', 'score_math'])

    # Drop 'ladder, 'schooldistrict_id'
    df = df.drop(columns=['ladder', 'schooldistrict_id'], errors='ignore')


    # Fix numeric column: experience
    df['experience'] = df['experience'].fillna(df['experience'].median())

    # Fix ethnicity BEFORE filling Unknown
    df['ethnicity'] = df['ethnicity'].apply(
        lambda x: x if x in ['cauc', 'afam'] else 'other'
    )

    # Replace missing categorical values
    df = df.fillna({
        'ethnicity': 'other',
        'lunch': 'Unknown',
        'birth': 'Unknown',
        'class_type': 'Unknown',
        't_ethnicity': 'Unknown',
        'degree': 'Unknown'
    })

    # Fix degree categories
    df['degree'] = df['degree'].apply(
        lambda x: x if x in ['bachelor', 'master'] else 'other'
    )
    return df


def remove_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return df[(df[col] >= lower) & (df[col] <= upper)]


df_clean = clean_data(df.copy())

for col in ['score_read', 'score_math', 'experience']:
    df_clean = remove_outliers_iqr(df_clean, col)

#### Splitting, Scaling and One-Hot Encoding

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer

X = df_clean.drop(columns=['score_read', 'score_math'])
y_read = df_clean['score_read']
y_math = df_clean['score_math']

X_train, X_test, y_read_train, y_read_test, y_math_train, y_math_test = train_test_split(
    X, y_read, y_math, test_size=0.2, random_state=42
)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

preprocessor = make_column_transformer(
    (StandardScaler(), num_cols),
    (OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
)

#### LightGBM Baseline

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# reading model
model_read = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(random_state=42)),
])

model_read.fit(X_train, y_read_train)
yhat_read_test = model_read.predict(X_test)

# math model
model_math = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(random_state=42)),
])

model_math.fit(X_train, y_math_train)
yhat_math_test = model_math.predict(X_test)

# MSE
sq_err_read = (yhat_read_test - y_read_test) ** 2
sq_err_math = (yhat_math_test - y_math_test) ** 2

MSE_read = sq_err_read.mean()
MSE_math = sq_err_math.mean()

n_test = len(X_test)
MSE_o = (sq_err_read + sq_err_math).sum() / (2.0 * n_test)

print(f"MSE_read: {MSE_read:.4f}")
print(f"MSE_math: {MSE_math:.4f}")
print(f"MSE_o: {MSE_o:.4f}")

MSE_read: 477.5372
MSE_math: 1507.9873
MSE_o: 992.7622


c:\Users\huiqi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\huiqi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


: 

#### LightGBM Tuned

In [5]:
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# LightGBM params
params = dict(
    colsample_bytree=0.8,
    learning_rate=0.03,
    n_estimators=500,
    num_leaves=31,
    subsample=1.0,
    random_state=42,
    verbose=-1,
)

# reading model
model_read = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(**params)),
])

model_read.fit(X_train, y_read_train)
yhat_read_test = model_read.predict(X_test)

# math model
model_math = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(**params)),
])

model_math.fit(X_train, y_math_train)
yhat_math_test = model_math.predict(X_test)

# --- MSEs (same as you had) ---
sq_err_read = (yhat_read_test - y_read_test) ** 2
sq_err_math = (yhat_math_test - y_math_test) ** 2

MSE_read = sq_err_read.mean()
MSE_math = sq_err_math.mean()

n_test = len(X_test)
MSE_o = (sq_err_read + sq_err_math).sum() / (2.0 * n_test)

print(f"MSE_read: {MSE_read:.4f}")
print(f"MSE_math: {MSE_math:.4f}")
print(f"MSE_o: {MSE_o:.4f}")


c:\Users\huiqi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MSE_read: 467.4188
MSE_math: 1514.1838
MSE_o: 990.8013


c:\Users\huiqi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
